<a href="https://colab.research.google.com/github/aish466-p/aishwarya/blob/main/NLP_DATASET_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_excel("hospital.xlsx")

print("Dataset shape:", df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())

Dataset shape: (996, 3)
                                            Feedback  Sentiment Label  Ratings
0  Good and clean hospital. There is great team o...                1        5
1  Had a really bad experience during discharge. ...                1        5
2  I have visited to take my second dose and Proc...                1        4
3   That person was slightly clueless and offered...                1        3
4  There is great team of doctors and good OT fac...                0        1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 996 entries, 0 to 995
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Feedback         996 non-null    object
 1   Sentiment Label  996 non-null    int64 
 2   Ratings          996 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 23.5+ KB
None
Feedback           0
Sentiment Label    0
Ratings            0
dtype: int64


In [2]:
import re
import html

def clean_text(text):
    if pd.isna(text):
        return ""

    text = html.unescape(str(text))
    text = text.replace("\u00a0", " ")
    text = re.sub(r"Â", "", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s']", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.lower().strip()

df["Feedback_Clean"] = df["Feedback"].apply(clean_text)

# Remove empty feedback
df = df[df["Feedback_Clean"] != ""]

# Remove duplicate cleaned feedback
df = df.drop_duplicates(subset=["Feedback_Clean"])

print(df[["Feedback", "Feedback_Clean"]].head())
print("Cleaned dataset shape:", df.shape)

                                            Feedback  \
0  Good and clean hospital. There is great team o...   
1  Had a really bad experience during discharge. ...   
2  I have visited to take my second dose and Proc...   
3   That person was slightly clueless and offered...   
4  There is great team of doctors and good OT fac...   

                                      Feedback_Clean  
0  good and clean hospital there is great team of...  
1  had a really bad experience during discharge t...  
2  i have visited to take my second dose and proc...  
3  that person was slightly clueless and offered ...  
4  there is great team of doctors and good ot fac...  
Cleaned dataset shape: (932, 4)


In [3]:
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    words = word_tokenize(text)

    words = [
        word for word in words
        if word.isalpha() and word not in stop_words
    ]

    return words

df["Tokens"] = df["Feedback_Clean"].apply(
    preprocess_text
)

print(df[["Feedback_Clean", "Tokens"]].head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...


                                      Feedback_Clean  \
0  good and clean hospital there is great team of...   
1  had a really bad experience during discharge t...   
2  i have visited to take my second dose and proc...   
3  that person was slightly clueless and offered ...   
4  there is great team of doctors and good ot fac...   

                                              Tokens  
0  [good, clean, hospital, great, team, doctors, ...  
1  [really, bad, experience, discharge, need, sen...  
2  [visited, take, second, dose, process, really,...  
3  [person, slightly, clueless, offered, one, pac...  
4         [great, team, doctors, good, ot, facility]  


[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def apply_stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

df["Stemmed_Text"] = df["Tokens"].apply(
    apply_stemming
)

print(df[["Tokens", "Stemmed_Text"]].head())

                                              Tokens  \
0  [good, clean, hospital, great, team, doctors, ...   
1  [really, bad, experience, discharge, need, sen...   
2  [visited, take, second, dose, process, really,...   
3  [person, slightly, clueless, offered, one, pac...   
4         [great, team, doctors, good, ot, facility]   

                                        Stemmed_Text  
0  [good, clean, hospit, great, team, doctor, goo...  
1  [realli, bad, experi, discharg, need, sensit, ...  
2  [visit, take, second, dose, process, realli, s...  
3  [person, slightli, clueless, offer, one, packa...  
4             [great, team, doctor, good, ot, facil]  


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=1000
)

X = vectorizer.fit_transform(
    df["Feedback_Clean"]
)

print("TF-IDF matrix shape:", X.shape)

print(vectorizer.get_feature_names_out()[:20])

TF-IDF matrix shape: (932, 1000)
['abhishek' 'able' 'about' 'above' 'absolutely' 'active' 'addresses'
 'admin' 'administration' 'administrative' 'admission' 'admit' 'admitted'
 'advance' 'advice' 'advised' 'afford' 'after' 'again' 'all']


In [6]:
df.to_excel(
    "hospital_cleaned_python.xlsx",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
